In [ ]:
# | default_exp features.fsc_gene

In [ ]:
# | export
from __future__ import annotations
import pandas as pd
import structlog
from kreview.eval_engine import FeatureEvaluator

log = structlog.get_logger()


In [ ]:
# | export
class FSCGeneEvaluator(FeatureEvaluator):
    """Extracts all gene-level fragment size characteristics."""

    name = "FSC_gene"
    source_file = ".FSC.gene.parquet"
    tier = 1
    category = "fragmentation"

    def extract(self, df: pd.DataFrame) -> dict[str, float]:
        extracted = {}
        try:
            if df.empty:
                return extracted

            cols = set(df.columns)
            ratios = [
                "ultra_short_ratio",
                "core_short_ratio",
                "mono_nucl_ratio",
                "di_nucl_ratio",
                "long_ratio",
            ]

            # Global aggregates
            for r in ratios:
                if r in cols:
                    extracted[f"global_{r}_mean"] = float(df[r].mean())

            if "normalized_depth" in cols:
                m_depth = df["normalized_depth"].mean()
                if pd.notna(m_depth) and m_depth > 0:
                    extracted["global_normalized_depth_cv"] = float(
                        df["normalized_depth"].std() / m_depth
                    )
                else:
                    extracted["global_normalized_depth_cv"] = 0.0

            # Per-gene extraction
            if "gene" in cols:
                for row in df.to_dict("records"):
                    g = str(row["gene"]).upper().replace(" ", "_").replace("-", "_")
                    if g == "NAN":
                        continue
                    for r in ratios:
                        if r in cols and pd.notna(row[r]):
                            extracted[f"{g}_{r}"] = float(row[r])
                    if "normalized_depth" in cols and pd.notna(row["normalized_depth"]):
                        extracted[f"{g}_normalized_depth"] = float(
                            row["normalized_depth"]
                        )

            return extracted
        except Exception as e:
            log.exception("extraction_failed", evaluator=self.name, error=str(e))
            return {}

In [ ]:
# | test
evaluator = FSCGeneEvaluator()
df_test = pd.DataFrame(
    [
        {
            "gene": "TP53",
            "ultra_short_ratio": 0.1,
            "core_short_ratio": 0.2,
            "normalized_depth": 1.2,
        }
    ]
)
res = evaluator.extract(df_test)
assert res["TP53_ultra_short_ratio"] == 0.1
assert "global_core_short_ratio_mean" in res